# Linear non-Gaussian models (DirectLiNGAM / ICA-LiNGAM)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Abel-ai-lab/andrey/blob/main/examples/lingam.ipynb)

When the mechanisms are linear and the noise is non-Gaussian, the data identify a single DAG, not
only its equivalence class. DirectLiNGAM finds a causal order one variable at a time with a
pairwise independence measure. ICA-LiNGAM runs independent component analysis (FastICA) and reads
the order from the unmixing matrix. Both return a fully oriented DAG, a causal order, and edge
weights.

## Start

Choose a CPU runtime and run this cell first. In Colab it installs `andrey-core==0.1.0` from PyPI;
local kernels use the installed environment.

In [ ]:
import os
import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    os.environ["ANDREY_DEVICE"] = "cpu"
    !pip install --quiet "andrey-core[numba,viz]==0.1.0" pandas

In [ ]:
import numpy as np
import pandas as pd
from IPython.display import display

import andrey
import andrey.metrics as metrics
import andrey.viz
from andrey.data import benchmarks

## Data

`lingam_sf` draws a scale-free graph with linear mechanisms and uniform noise: here 8 variables
and 2,000 samples.

In [ ]:
seed = 7
dataset = benchmarks.scm("lingam_sf", d=8).sample(n=2000, seed=seed, scale="standardize")
X, truth = dataset.data, dataset.observed_graph
print(f"{X.shape[0]} observations, {X.shape[1]} variables, seed {seed}")
display(andrey.viz.draw(truth))

## Fit both methods

DirectLiNGAM is deterministic. ICA-LiNGAM starts FastICA from a random point; `random_state` fixes
it, so a run repeats exactly.

In [ ]:
results = {
    "DirectLiNGAM": andrey.direct_lingam(X),
    "ICA-LiNGAM": andrey.ica_lingam(X, random_state=0),
}
for name, result in results.items():
    print(f"{name}: {result.structure.kind}, causal order {result.ordering}")
    display(andrey.viz.draw(result.structure))

## Edge weights

`weighted_adjacency[i, j]` is the coefficient of variable `i` in the equation for variable `j`, so
each nonzero entry is an edge `i -> j`.

In [ ]:
pd.DataFrame(results["ICA-LiNGAM"].weighted_adjacency).round(2)

## Check accuracy over ten datasets

SHD counts errors against the true DAG; lower is better. The empty graph's SHD is the number of
true edges. The table covers ten seeds and their median.

In [ ]:
methods = {
    "DirectLiNGAM": lambda X: andrey.direct_lingam(X),
    "ICA-LiNGAM": lambda X: andrey.ica_lingam(X, random_state=0),
}
rows = {}
for s in range(10):
    draw = benchmarks.scm("lingam_sf", d=8).sample(n=2000, seed=s, scale="standardize")
    d = draw.observed_graph.n_nodes
    empty = andrey.GraphStructure.from_numpy(np.zeros((d, d), dtype=np.uint8), kind="dag")
    rows[s] = {"Empty graph": metrics.score(empty, draw.observed_graph)["shd"]}
    for name, fit in methods.items():
        rows[s][name] = metrics.score(fit(draw.data), draw.observed_graph)["shd"]
shd = pd.DataFrame(rows)
shd.columns.name = "seed"
shd["median"] = shd.median(axis=1)
shd

## When to use them

Both methods rely on non-Gaussian noise. With Gaussian noise the order is not identified; PC, GES,
BOSS, or GRaSP return the CPDAG that the data do support. ICA-LiNGAM needs many samples: at
`n = 10d`, the size the [benchmarks](https://andrey.abel.ai/docs/benchmarks.html) use, it scores
worse than the empty graph, and this notebook uses `n = 250d`. Change `seed`, `d`, or `n` and rerun
to see how each method responds.